# Detección de Neumonía en Radiografías con Deep Learning y Random Forest
Este cuaderno compara tres enfoques para la detección automática de neumonía:
- Una red convolucional (CNN) entrenada desde cero.
- Un clasificador Random Forest sobre características extraídas por MobileNetV2.
- MobileNetV2 con fine-tuning y cabeza densa (transfer learning completo).

## 1. Importación de librerías necesarias

Importamos todas las librerías necesarias para el tratamiento de imágenes, construcción y entrenamiento de modelos con Keras, extracción de características con MobileNetV2, clasificación tradicional con Scikit-learn y visualización con matplotlib y seaborn.


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

## 2. Configuración de parámetros y rutas

Definimos las rutas a los conjuntos de datos (entrenamiento, validación y prueba), así como parámetros globales del proyecto como el tamaño de las imágenes, el batch size y el número de épocas de entrenamiento.


In [ ]:
data_dir = 'data'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'validation')
test_dir = os.path.join(data_dir, 'test')
img_width, img_height = 150, 150
batch_size = 32
epochs = 10

## 3. Generación de datos

Creamos generadores para cargar imágenes desde disco y procesarlas de forma automática.  
Aplicamos técnicas de aumento de datos para robustecer el modelo durante el entrenamiento, y normalizamos los píxeles dividiendo entre 255.  
Desactivamos el barajado (`shuffle=False`) en los conjuntos de entrenamiento y prueba para mantener el orden de las muestras, importante para la correspondencia con sus etiquetas.


In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir, target_size=(img_width, img_height),
    batch_size=batch_size, class_mode='binary', shuffle=False)
val_generator = val_datagen.flow_from_directory(val_dir, target_size=(img_width, img_height),
    batch_size=batch_size, class_mode='binary')
test_generator = val_datagen.flow_from_directory(test_dir, target_size=(img_width, img_height),
    batch_size=1, class_mode='binary', shuffle=False)

## 4. Entrenamiento o carga del modelo CNN

Si existe el archivo `modelo_neumonia.h5`, cargamos el modelo ya entrenado.  
En caso contrario, construimos una red neuronal convolucional desde cero, la entrenamos con los datos proporcionados y la guardamos para futuros usos.


In [ ]:
if os.path.exists('modelo_neumonia.h5'):
    print("Cargando modelo existente...")
    model = load_model('modelo_neumonia.h5')
else:
    print("Entrenando nuevo modelo CNN...")
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(img_width, img_height, 3)),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(0.0001), loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(train_generator, steps_per_epoch=train_generator.samples // batch_size,
        validation_data=val_generator, validation_steps=val_generator.samples // batch_size,
        epochs=epochs)
    model.save('modelo_neumonia.h5')

## 5. Predicción con la CNN

Realizamos predicciones sobre las imágenes del conjunto de prueba usando el modelo CNN.  
Convertimos las probabilidades en etiquetas binarias (0 o 1) comparándolas con un umbral de 0.5.


In [ ]:
predictions = model.predict(test_generator)
predicted_classes = (predictions > 0.5).astype(int)

## 6. Extracción de características con MobileNetV2

MobileNetV2 se utiliza como extractor de características sin reentrenamiento (transfer learning).  
A partir de cada imagen, genera un vector numérico que resume patrones visuales relevantes aprendidos en ImageNet.


In [ ]:
mobilenet = MobileNetV2(input_shape=(img_width, img_height, 3), include_top=False,
                          weights='imagenet', pooling='avg')
# Guardamos el extractor para que la app pueda usarlo con el modelo RF
mobilenet.save('modelo_mobilenet_features.h5')

train_features = mobilenet.predict(train_generator)
test_features = mobilenet.predict(test_generator)

## 7. Clasificación con Random Forest

Entrenamos un modelo `RandomForestClassifier` utilizando los vectores de características generados por MobileNetV2.  
Activamos `class_weight='balanced'` para compensar el desbalance entre clases.  
Después usamos este modelo para predecir sobre las imágenes de prueba.


In [ ]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(train_features, train_generator.classes)
y_pred_rf = rf.predict(test_features)

# Guardamos el modelo RF para usarlo en la app
joblib.dump(rf, 'modelo_rf.pkl')

## 8. Modelo 3: MobileNetV2 Fine-tuned

Tercer enfoque: usamos MobileNetV2 como base pero añadimos una cabeza densa y la entrenamos de extremo a extremo.

- **Fase 1:** La base de MobileNetV2 está congelada. Solo entrenamos la cabeza (Dense + Dropout).
- **Fase 2:** Descongelamos las últimas 20 capas y ajustamos con learning rate muy bajo (`1e-5`) para no destruir los pesos de ImageNet.

Esto se distingue del enfoque anterior (Modelo 2) en que el clasificador final es también una red neuronal, no un algoritmo clásico.

In [ ]:
if os.path.exists('modelo_mobilenet_finetuned.h5'):
    print("Cargando modelo MobileNetV2 fine-tuned existente...")
    model_ft = load_model('modelo_mobilenet_finetuned.h5')
else:
    print("Fase 1: entrenando cabeza densa (base congelada)...")
    base = MobileNetV2(input_shape=(img_width, img_height, 3), include_top=False, weights='imagenet')
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    model_ft = Model(inputs=base.input, outputs=output)
    model_ft.compile(optimizer=Adam(0.0001), loss='binary_crossentropy', metrics=['accuracy'])
    model_ft.fit(train_generator, steps_per_epoch=train_generator.samples // batch_size,
        validation_data=val_generator, validation_steps=val_generator.samples // batch_size,
        epochs=epochs)

    print("Fase 2: fine-tuning últimas 20 capas...")
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    model_ft.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
    model_ft.fit(train_generator, steps_per_epoch=train_generator.samples // batch_size,
        validation_data=val_generator, validation_steps=val_generator.samples // batch_size,
        epochs=5)
    model_ft.save('modelo_mobilenet_finetuned.h5')

preds_ft = model_ft.predict(test_generator)
y_pred_ft = (preds_ft > 0.5).astype(int).flatten()
y_true = test_generator.classes
print("Predicciones MobileNetV2 Fine-tuned completadas.")

## 9. Comparación de resultados entre los tres modelos

Calculamos métricas de rendimiento para los tres enfoques, incluyendo:
- Exactitud global (`accuracy`)
- **Precisión**: de los casos detectados como neumonía, ¿cuántos eran realmente neumonía?
- **Recall**: de todos los casos de neumonía reales, ¿cuántos detectamos?
- F1-Score (media armónica de precisión y recall)

En contexto médico, el **recall** es la métrica más importante: minimizar falsos negativos (no detectar una neumonía real) es crítico.

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy":  round(float(accuracy_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "recall":    round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "f1":        round(float(f1_score(y_true, y_pred, zero_division=0)), 4)
    }

metrics = {
    "CNN":                    compute_metrics(y_true, predicted_classes),
    "MobileNetV2 + RF":       compute_metrics(y_true, y_pred_rf),
    "MobileNetV2 Fine-tuned": compute_metrics(y_true, y_pred_ft)
}

# Guardar métricas para la app Streamlit
with open('model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"{'Modelo':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 65)
for name, m in metrics.items():
    print(f"{name:<25} {m['accuracy']:>10.2%} {m['precision']:>10.2%} {m['recall']:>10.2%} {m['f1']:>10.2%}")

print("\n=== Informes detallados ===")
print("--- CNN ---")
print(classification_report(y_true, predicted_classes, target_names=['Normal', 'Neumonía']))
print("--- MobileNetV2 + RF ---")
print(classification_report(y_true, y_pred_rf, target_names=['Normal', 'Neumonía']))
print("--- MobileNetV2 Fine-tuned ---")
print(classification_report(y_true, y_pred_ft, target_names=['Normal', 'Neumonía']))

## 10. Visualización de matrices de confusión

Mostramos las matrices de confusión para los tres modelos.  
Permite identificar dónde falla cada uno: falsos positivos (detecta neumonía cuando no hay) vs. falsos negativos (no detecta neumonía cuando sí hay).  
En diagnóstico médico los **falsos negativos son más graves**: el coste de no detectar una neumonía real supera al de un falso positivo.

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Neumonía'],
                yticklabels=['Normal', 'Neumonía'])
    plt.title(title)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.show()

plot_confusion(y_true, predicted_classes, "Matriz de confusión - CNN")
plot_confusion(y_true, y_pred_rf,         "Matriz de confusión - MobileNetV2 + RF")
plot_confusion(y_true, y_pred_ft,         "Matriz de confusión - MobileNetV2 Fine-tuned")